# NB31: recover ordering among weak CYP2D6 compounds

**Hypothesis:** fixed 3.5 pseudo-targets erase useful ordering among weak compounds. NB30 found extra weak-screen ordering, but its global blend lost potency accuracy. Train that ordering into a potency model, then apply only the difference from a matched fixed-target control in the predicted-weak region.

Four matched arms: measured-only, fixed 3.5 pseudo-target, narrow graded pseudo-target (3.25–3.75), and wider graded pseudo-target (3.0–4.0). Graded targets follow the empirical ordering of observed primary-screen response **only inside the training split**. These values are hypotheses, not measured pIC50s or certified upper bounds. The primary screen used TDI conditions; its relationship to direct potency may fail.

28 fits: five scaffold folds, holdout and blind fit for each arm. Eight TabPFN estimators per fit. Same seeds, inputs and sampled weak compounds across pseudo arms. Features: frozen PCA200 + public qHTS5. Historical auxiliary screen predictions are excluded. Reuses saved NB30 fold IDs to avoid library-dependent split changes. All evaluation compounds' screen and potency labels are excluded from their model training and pseudo-target construction.

The correction is `0.85 × gate × (graded − fixed)` added to v31's raw CYP2D6 blend; gate is 1 below control prediction 3.75 and falls linearly to 0 at 4.5. No new blend-weight search. Candidate means and spreads are matched to the v31 reference. The measured-only arm diagnoses the cost of pseudo-labeling.

**Decision rule fixed before training:** choose on development only, requiring contextual weak-screen correlation gain ≥0.02, labelled-potency correlation loss ≤0.005 and labelled ST-RAE increase ≤0.005 versus v31. Require improvement in at least 4/5 development folds for the weak-screen diagnostic. Freeze this choice before revealing holdout results. A holdout failure means no promotion; no alternative is selected on holdout.

The NB24 weak-screen comparator has legacy feature contamination and is only contextual. The matched NB31 fixed-vs-graded comparison is cleaner. The full v31 dense-population prediction is unavailable for unlabelled training compounds; do not describe the component screen diagnostic as full-v31 validation. The existing project holdout is repeatedly reused. This experiment cannot prove a leaderboard breakthrough.

Source: [official assay design](https://openadmet.ghost.io/announcing-openadmets-cyp-inhibition-blind-challenge/). **Keep submission v31.** Notebook number NB31 is separate from submission version v31. This notebook saves results for review and does not generate or upload a submission.


In [ ]:
RUN = 'a'
ISO = 'CYP2D6'
N_ESTIMATORS = 8
PRED_BATCH = 256
SEED = 42
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'
OUTDIR = '/content/drive/MyDrive/cyp_results'
from google.colab import drive
drive.mount('/content/drive')
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)


In [ ]:
%pip -q install tabpfn==2.2.1


In [ ]:
IMPLEMENTATION_SHA = '385ad92e1ece875a23965d8c2480d0024a63d4ce65b872235033f197b06157b6'
import gc, hashlib, json, os, platform, tarfile, time, uuid
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from scipy.stats import rankdata, spearmanr
from sklearn.model_selection import GroupKFold
from tabpfn import TabPFNRegressor

CACHE = Path('/content/nb31_cache')
OUT = Path(OUTDIR)
CACHE.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
assert Path('/content/drive/MyDrive').exists(), 'Mount Drive first.'
NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy', 'qhts_tr.npy', 'qhts_te.npy']
with tarfile.open(PAYLOAD_FILE) as archive:
    # Extract only named ordinary payload files, without accepting arbitrary archive paths.
    for name in NEED:
        matches = [m for m in archive.getmembers() if m.name in (name, './'+name)]
        assert len(matches) == 1 and matches[0].isfile(), f'Missing or invalid payload member: {name}'
        (CACHE / name).write_bytes(archive.extractfile(matches[0]).read())

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy').astype(bool)
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=False)
PCA_FILE = OUT / 'nb24_pca_surrogate_adme.npz'
assert PCA_FILE.exists(), 'Run NB24 surrogate PCA preparation first; its saved PCA file is required.'
with np.load(PCA_FILE, allow_pickle=False) as p:
    PF, PF_TE = p['tr'], p['te']
Q, Q_TE = np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'qhts_te.npy')
X, X_TE = np.hstack([PF, Q]).astype(np.float32), np.hstack([PF_TE, Q_TE]).astype(np.float32)
N, NTE = len(meta), len(mete)
assert len(X) == N == len(hold) == len(G) and len(X_TE) == NTE
assert X.shape[1] == X_TE.shape[1] == 205 and PF.shape[1] == 200
assert np.isfinite(X).all() and np.isfinite(X_TE).all(), 'Unexpected nonfinite feature values.'
assert meta['Molecule_Name'].is_unique and mete['Molecule_Name'].is_unique
assert set(meta['Molecule_Name']).isdisjoint(set(mete['Molecule_Name']))
COL = f'{ISO}_pIC50_direct_inhibition'
Y = meta[COL].to_numpy(float)
SCREEN = -meta[f'{ISO}_log2fc'].to_numpy(float)
SCREEN_OBS, LABELLED = np.isfinite(SCREEN), np.isfinite(Y)
ISO_ALL = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
obs_any = np.isfinite(meta[[f'{i}_pIC50_direct_inhibition' for i in ISO_ALL]].to_numpy(float)).any(1)
dev_all = np.flatnonzero(obs_any & ~hold)
ho_all = np.flatnonzero(obs_any & hold)
assert len(dev_all) + len(ho_all) == N
assert set(G[dev_all]).isdisjoint(set(G[ho_all])), 'Holdout scaffold groups overlap development.'
FOLDS = list(GroupKFold(5).split(dev_all, groups=G[dev_all]))
for tr, va in FOLDS:
    assert set(G[dev_all[tr]]).isdisjoint(set(G[dev_all[va]]))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu': torch.set_num_threads(4)

BASE_FILES = [OUT / f'nb24_{r}_pseudo_{ISO}.npz' for r in ['a', 'b', 'c', 'e']]
for p in BASE_FILES: assert p.exists(), f'Comparator missing: {p.name}'
BASE = {}
for p in BASE_FILES:
    with np.load(p, allow_pickle=False) as z:
        assert np.array_equal(z['dev'], np.flatnonzero(LABELLED & ~hold))
        assert np.array_equal(z['ho'], np.flatnonzero(LABELLED & hold))
        assert z['oof_full'].shape == z['hp_full'].shape == (N,) and z['test'].shape == (NTE,)
        for key in ['oof_full', 'hp_full', 'test']:
            BASE.setdefault(key, []).append(z[key].copy())
BASE = {k: np.mean(v, axis=0) for k, v in BASE.items()}
assert np.isfinite(BASE['oof_full'][dev_all]).all() and np.isfinite(BASE['hp_full'][ho_all]).all() and np.isfinite(BASE['test']).all()

def file_sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def array_sha(a): return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()

# Use the exact saved NB30 evaluation folds, with all training membership revalidated.
FOLDS = []
for k in range(5):
    with np.load(OUT / f'nb30_a_CYP2D6_screen_f{k}.npz', allow_pickle=False) as z:
        ev = z['eval_rows'].copy()
    assert len(np.unique(ev)) == len(ev) and np.isin(ev, dev_all).all()
    va = np.flatnonzero(np.isin(dev_all, ev)); tr = np.flatnonzero(~np.isin(dev_all, ev))
    assert np.array_equal(dev_all[va], ev)
    assert set(G[dev_all[tr]]).isdisjoint(set(G[ev]))
    FOLDS.append((tr, va))
assert np.array_equal(np.sort(np.concatenate([dev_all[va] for _,va in FOLDS])), dev_all)
ARMS = {'measured':None, 'fixed':0.0, 'graded025':0.25, 'graded050':0.5}
import io, base64
REF_B64 = '...'
with np.load(io.BytesIO(base64.b64decode(REF_B64)), allow_pickle=False) as z:
    REF = {k:z[k].copy() for k in z.files}
assert np.array_equal(REF['dev'],np.flatnonzero(LABELLED & ~hold))
assert np.array_equal(REF['ho'],np.flatnonzero(LABELLED & hold))
def match_moments(p, ref):
    assert np.isfinite(p).all() and np.std(p)>1e-10
    return np.mean(ref)+(p-np.mean(p))/np.std(p)*np.std(ref)
assert np.max(np.abs(match_moments(REF['test'],REF['submitted'])-REF['submitted']))<3e-6
CONFIG = dict(notebook='nb31_ordered_weak_v1',run=RUN,seed=SEED,n_estimators=N_ESTIMATORS,
              pred_batch=PRED_BATCH,arms=ARMS,weak_cut_percentile=5,weak_sample_ratio=1.0,
              pseudo_center=3.5,gate_full=3.75,gate_zero=4.5,component_weight=0.85,
              features='PCA200_qHTS5_no_aux',inputs={n:file_sha(CACHE/n) for n in NEED},
              pca_sha=file_sha(PCA_FILE),x_sha=array_sha(X),xt_sha=array_sha(X_TE),
              baseline_sha={p.name:file_sha(p) for p in BASE_FILES},reference_sha=hashlib.sha256(base64.b64decode(REF_B64)).hexdigest(),
              fold_sha=[array_sha(dev_all[va]) for _,va in FOLDS],
              versions={k:version(k) for k in ['tabpfn','torch','numpy','scikit-learn']},device=DEVICE,
              selection=dict(min_weak_rho_gain=.02,max_potency_rho_loss=.005,max_strae_increase=.005,min_positive_folds=4),
              implementation_sha=IMPLEMENTATION_SHA)
assert CONFIG['versions']['tabpfn']=='2.2.1'
SIG=hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest();CONFIG['signature']=SIG
PREFIX=f'nb31_{RUN}'
def save_json(path,data):
    tmp=Path(str(path)+'.'+uuid.uuid4().hex+'.tmp');tmp.write_text(json.dumps(data,indent=2,allow_nan=False));os.replace(tmp,path)
def save_npz(path,**kw):
    tmp=Path(str(path)+'.'+uuid.uuid4().hex+'.tmp.npz');np.savez_compressed(tmp,**kw);os.replace(tmp,path)
cp=OUT/f'{PREFIX}_config.json'
if cp.exists(): assert json.loads(cp.read_text())==CONFIG, 'Changed inputs or settings: use a new RUN letter.'
else: save_json(cp,CONFIG)
print('Ready:',len(dev_all),'development,',len(ho_all),'holdout,',NTE,'blind; 28 fits. Signature',SIG[:12])


In [ ]:
def make_training(compounds, amplitude, seed):
    compounds=np.asarray(compounds,dtype=int)
    real=compounds[LABELLED[compounds]]
    if amplitude is None: return real,Y[real].copy(),dict(n_real=len(real),n_weak=0,cut=None)
    paired=real[SCREEN_OBS[real]]
    assert len(paired)>20
    cut=float(np.percentile(SCREEN[paired],5))
    eligible=compounds[~LABELLED[compounds]&SCREEN_OBS[compounds]&(SCREEN[compounds]<cut)]
    assert len(eligible)>0
    rng=np.random.default_rng(seed)
    chosen=np.sort(rng.choice(eligible,min(len(real),len(eligible)),replace=False))
    # Reference ranks are computed from all eligible TRAINING compounds before subsampling.
    ranks=(rankdata(SCREEN[eligible])-.5)/len(eligible)
    values=3.5+float(amplitude)*(2*ranks-1)
    weak_y=values[np.searchsorted(eligible,chosen)]
    rows=np.r_[real,chosen];target=np.r_[Y[real],weak_y]
    assert len(np.unique(rows))==len(rows) and np.isfinite(target).all()
    return rows,target,dict(n_real=len(real),n_weak=len(chosen),cut=cut,
        weak_min=float(weak_y.min()),weak_max=float(weak_y.max()),weak_mean=float(weak_y.mean()))

def fit_stage(arm,stage,compounds,ev,eval_x,seed):
    rows,target,manifest=make_training(compounds,ARMS[arm],seed)
    ev=np.asarray(ev,dtype=int)
    if stage!='test':
        assert not np.intersect1d(rows,ev).size
        assert set(G[rows]).isdisjoint(set(G[ev]))
        assert not hold[rows].any()
    name=f'{PREFIX}_{arm}_{stage}'
    final=OUT/(name+'.npz');partial=OUT/(name+'_partial.npz')
    manifest.update(signature=SIG,arm=arm,stage=stage,rows_sha=array_sha(rows),target_sha=array_sha(target),eval_sha=array_sha(ev))
    mp=OUT/(name+'_training.json')
    if mp.exists():assert json.loads(mp.read_text())==manifest
    else:save_json(mp,manifest)
    def load_checked(path):
        with np.load(path,allow_pickle=False) as z:
            assert str(z['signature'])==SIG and str(z['target_sha'])==array_sha(target)
            assert np.array_equal(z['train_rows'],rows) and np.array_equal(z['eval_rows'],ev)
            p=z['pred'].copy();assert p.shape==(len(ev),)
            return p
    if final.exists():
        p=load_checked(final);assert np.isfinite(p).all();print(arm,stage,'complete; skipped',flush=True);return p
    p=load_checked(partial) if partial.exists() else np.full(len(ev),np.nan)
    model=None;start=time.time()
    try:
        model=TabPFNRegressor(device=DEVICE,n_estimators=N_ESTIMATORS,random_state=seed,
            ignore_pretraining_limits=True,fit_mode='low_memory',memory_saving_mode=True,n_jobs=1)
        model.fit(X[rows],target)
        for lo in range(0,len(ev),PRED_BATCH):
            hi=min(lo+PRED_BATCH,len(ev))
            if np.isfinite(p[lo:hi]).all():continue
            got=np.asarray(model.predict(eval_x[lo:hi]),dtype=float).reshape(-1)
            assert got.shape==(hi-lo,) and np.isfinite(got).all();p[lo:hi]=got
            save_npz(partial,signature=np.array(SIG),target_sha=np.array(array_sha(target)),train_rows=rows,eval_rows=ev,pred=p)
            print(arm,stage,f'{np.isfinite(p).sum()}/{len(p)} saved; {(time.time()-start)/60:.1f} min',flush=True)
        save_npz(final,signature=np.array(SIG),target_sha=np.array(array_sha(target)),train_rows=rows,eval_rows=ev,pred=p)
        return p
    finally:
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()

def gate(p):return np.clip((4.5-p)/.75,0,1)
def correction(fixed,graded):return gate(fixed)*(graded-fixed)
OOF={}
for arm in ARMS:
    pred=np.full(N,np.nan)
    for k,(tr,va) in enumerate(FOLDS):
        pred[dev_all[va]]=fit_stage(arm,f'f{k}',dev_all[tr],dev_all[va],X[dev_all[va]],SEED+k)
    assert np.isfinite(pred[dev_all]).all();OOF[arm]=pred
print('All development predictions complete. Next cell freezes selection before holdout fitting.')


In [ ]:
def correlation(y, p):
    ok = np.isfinite(y) & np.isfinite(p)
    if ok.sum() < 3 or np.std(y[ok]) == 0 or np.std(p[ok]) == 0: return None
    return float(spearmanr(y[ok], p[ok]).statistic)
def rank_percentile(p): return (rankdata(p) - 0.5) / len(p)
def rank_mix(base, screen, w=0.2): return (1-w)*rank_percentile(base) + w*rank_percentile(screen)
def auc(p, pos, neg):
    if not len(pos) or not len(neg): return None
    r = rankdata(np.r_[p[pos], p[neg]])
    return float((r[:len(pos)].sum()-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg)))
# Define clean blanks using only development-labelled screen values.
cut = float(np.percentile(SCREEN[~hold & LABELLED & SCREEN_OBS], 5))
CLEAN = ~LABELLED & SCREEN_OBS & (SCREEN < cut)
def metrics(pred, idx):
    screened = idx[SCREEN_OBS[idx]]
    blank = idx[SCREEN_OBS[idx] & ~LABELLED[idx]]
    labelled = idx[LABELLED[idx]]
    labelled_screen = idx[SCREEN_OBS[idx] & LABELLED[idx]]
    clean = idx[CLEAN[idx]]
    return dict(screen_all_rho=correlation(SCREEN[screened], pred[screened]),
                screen_blank_rho=correlation(SCREEN[blank], pred[blank]),
                screen_labelled_rho=correlation(SCREEN[labelled_screen], pred[labelled_screen]),
                labelled_potency_rho=correlation(Y[labelled], pred[labelled]),
                label_clean_blank_auc=auc(pred, labelled_screen, clean),
                n_screen=len(screened), n_blank=len(blank), n_labelled=len(labelled), n_clean_blank=len(clean))

LO=meta[COL+'_conf_low'].to_numpy(float);HI=meta[COL+'_conf_high'].to_numpy(float)
def soft(p,lo,hi):return np.maximum(lo-p,0)+np.maximum(p-hi,0)
def strae(y,p,lo,hi):
    den=soft(np.full_like(y,np.mean(y)),lo,hi).sum()
    return float(soft(p,lo,hi).sum()/den) if den>0 else None
# Calibrate the raw v31 baseline on development only; no holdout outcome moments.
idx=REF['dev'];yd=Y[idx];raw=REF['oof'];scale=float(np.clip(np.corrcoef(yd,raw)[0,1],.3,.95)*np.std(yd)/np.std(raw))
shifts=np.arange(-.6,.81,.05)
shift=float(min(shifts,key=lambda d:strae(yd,yd.mean()+d+(raw-raw.mean())*scale,LO[idx],HI[idx])))
intercept=float(yd.mean()+shift-raw.mean()*scale)
def calibrated(p):return intercept+scale*p
ref_dev=calibrated(REF['oof'])
reference_metrics=metrics(BASE['oof_full'],dev_all)
base_potency=correlation(yd,REF['oof']);base_strae=strae(yd,ref_dev,LO[idx],HI[idx])
DEV={};ELIGIBLE=[]
for arm in ARMS:
    DEV[arm]={'standalone':metrics(OOF[arm],dev_all)}
    if not arm.startswith('graded'):continue
    delta=correction(OOF['fixed'],OOF[arm]);component=BASE['oof_full']+delta
    cand=match_moments(calibrated(REF['oof']+.85*delta[idx]),ref_dev)
    mm=metrics(component,dev_all)
    weak_gain=mm['screen_blank_rho']-reference_metrics['screen_blank_rho']
    pot=correlation(yd,cand);st=strae(yd,cand,LO[idx],HI[idx])
    gains=[metrics(component,dev_all[va])['screen_blank_rho']-metrics(BASE['oof_full'],dev_all[va])['screen_blank_rho'] for _,va in FOLDS]
    matched_gain=metrics(OOF['fixed']+delta,dev_all)['screen_blank_rho']-metrics(OOF['fixed'],dev_all)['screen_blank_rho']
    passed=bool(weak_gain>=.02 and pot>=base_potency-.005 and st<=base_strae+.005 and sum(g>0 for g in gains)>=4 and matched_gain>0)
    DEV[arm].update(contextual_component=mm,weak_gain=weak_gain,matched_weak_gain=matched_gain,
        fold_weak_gains=gains,v31_labelled_rho=pot,v31_labelled_strae=st,eligible=passed)
    if passed:ELIGIBLE.append(arm)
CHOSEN=max(ELIGIBLE,key=lambda a:DEV[a]['weak_gain']) if ELIGIBLE else None
SELECTION=dict(signature=SIG,chosen=CHOSEN,development=DEV,baseline_rho=base_potency,baseline_strae=base_strae,
               calibration=dict(scale=scale,intercept=intercept),rule=CONFIG['selection'])
sp=OUT/f'{PREFIX}_selection.json'
if sp.exists():assert json.loads(sp.read_text())==SELECTION,'Selection changed; use a new run.'
else:save_json(sp,SELECTION)
print(json.dumps(SELECTION,indent=2));print('Frozen development choice:',CHOSEN,'(None means no candidate passed.)')


In [ ]:
HP={};TEST={}
for arm in ARMS:
    hp=np.full(N,np.nan)
    hp[ho_all]=fit_stage(arm,'hold',dev_all,ho_all,X[ho_all],SEED+100)
    test=fit_stage(arm,'test',np.arange(N),np.arange(NTE),X_TE,SEED+200)
    HP[arm]=hp;TEST[arm]=test
    save_npz(OUT/f'{PREFIX}_{arm}_CYP2D6.npz',signature=np.array(SIG),oof_full=OOF[arm],hp_full=hp,
        oof=OOF[arm][REF['dev']],dev=REF['dev'],hold=hp[REF['ho']],ho=REF['ho'],test=test,
        dev_all=dev_all,ho_all=ho_all)
RESULTS=dict(config=CONFIG,selection=SELECTION,holdout={a:metrics(HP[a],ho_all) for a in ARMS},
    conclusion='No candidate passed development selection.',
    limitations=['Weak pseudo-targets are hypothetical direct-potency values derived from TDI-condition screen order.',
      'NB24 component dense-screen comparison is contextual, not clean full-v31 validation.',
      'Repeatedly reused holdout and candidate selection limit uncertainty estimates. No automatic submission.'])
if CHOSEN is not None:
    idx=REF['ho'];delta=correction(HP['fixed'],HP[CHOSEN]);component=BASE['hp_full']+delta
    ref=calibrated(REF['hold']);cand=match_moments(calibrated(REF['hold']+.85*delta[idx]),ref)
    before=metrics(BASE['hp_full'],ho_all);after=metrics(component,ho_all)
    rho_before=correlation(Y[idx],ref);rho_after=correlation(Y[idx],cand)
    st_before=strae(Y[idx],ref,LO[idx],HI[idx]);st_after=strae(Y[idx],cand,LO[idx],HI[idx])
    dh=after['screen_blank_rho']-before['screen_blank_rho']
    matched=metrics(HP['fixed']+delta,ho_all)['screen_blank_rho']-metrics(HP['fixed'],ho_all)['screen_blank_rho']
    gate_pass=bool(dh>0 and matched>0 and rho_after>=rho_before-.005 and st_after<=st_before+.005)
    # A conditional scaffold interval; it does not account for experiment selection.
    ix=ho_all[SCREEN_OBS[ho_all]&~LABELLED[ho_all]];groups=[ix[G[ix]==g] for g in np.unique(G[ix])]
    rng=np.random.default_rng(31);boot=[]
    for _ in range(2000):
        jj=np.concatenate([groups[g] for g in rng.integers(0,len(groups),len(groups))])
        boot.append(correlation(SCREEN[jj],component[jj])-correlation(SCREEN[jj],BASE['hp_full'][jj]))
    raw_test=REF['test']+.85*correction(TEST['fixed'],TEST[CHOSEN])
    test_candidate=match_moments(raw_test,REF['submitted'])
    RESULTS['candidate']=dict(arm=CHOSEN,holdout_gate_pass=gate_pass,component_before=before,component_after=after,
        contextual_weak_delta=dh,matched_weak_delta=matched,contextual_weak_scaffold_ci=np.quantile(boot,[.025,.975]).tolist(),
        v31_rho_before=rho_before,v31_rho_after=rho_after,v31_strae_before=st_before,v31_strae_after=st_after,
        blind_rank_correlation=correlation(REF['submitted'],test_candidate),blind_mean_abs_change=float(np.mean(abs(test_candidate-REF['submitted']))))
    save_npz(OUT/f'{PREFIX}_candidate_review_only.npz',signature=np.array(SIG),test=test_candidate,
             hold=cand,ho=idx,dev=REF['dev'],arm=np.array(CHOSEN),holdout_gate_pass=np.array(gate_pass))
    RESULTS['conclusion']='Passed provisional gates; review results before any submission.' if gate_pass else 'Holdout rejected the frozen candidate. Keep v31.'
save_json(OUT/f'results_{PREFIX}.json',RESULTS)
print(json.dumps(RESULTS,indent=2))


In [ ]:
import zipfile
archive_path=Path(f'/content/{PREFIX}_results.zip')
files=[p for p in OUT.glob(f'{PREFIX}_*') if p.suffix in ['.json','.npz'] and '_partial' not in p.name]
files.append(OUT/f'results_{PREFIX}.json')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(set(files)):z.write(p,p.name)
from IPython.display import FileLink,display
display(FileLink(str(archive_path)))
print('Results and resumable prediction batches are saved in Drive/cyp_results.')
